In [1]:
from premier_rapport import premier_rapport
import pandas as pd

# import communs  : description des fichiers, paramètres,...
from communs import rep_src, rep_dst, fichiers_annuels

actions = []

def log_action (action) :
    print (" - ", action)
    actions.append(action)
    pass

dfrub = {}

# Les descritions de fichiers sont contenues dans un dictionnaire appelé
# fichiers_annuels. Il contient quatre rubriques, chaque rubrique contient
# la liste des fichiers. Chaque fichier il y a le nom du fichier, le séparateur
# la phaseet quelques infos. La phase 1 correspond au fichiers de 2005 à 2018,
# la phase 2 de 2019 à 2022.
# Ici nous sélectionnons les fichier de la phase 2 uniquement.

log_action("Chargement des fichiers")
for key, value in fichiers_annuels.items():  # Pour chaque rubrique
    # nb_obs sert à l'affichage du nombre d'observations (lignes, hors entêtes)
    # Pour mise au point et contrôle.
    nb_obs = 0

    print ()
    print("Rubrique : ", key)
    # dfl est une liste, elle contient les DataFrames de chaque fichier annuel
    # Elle sert ultérieurement à la concaténation.
    dfl = []
    
    # réccupération des informations de conversions de type 
    dtype = value.get("dtypes")

    for fichier_origine in value["fichiers"]:  # Pour chaque fichier annuel
        # Phase 2 : 2019 --> 2022 : 4 années
        if fichier_origine.get("phase") == "2":
            nom_fichier = fichier_origine["nom"]
            # lecture du fichier
            df = pd.read_csv(rep_src + nom_fichier,
                             sep=fichier_origine["sep"],
                             dtype=dtype,
                             encoding="latin_1",
                             index_col=False,
                             quotechar="\"",
                             low_memory=False)
            
            nb_obs += df.shape[0]
            # Il y a un cas de renommage de colonne à faire 
            if fichier_origine.get("rename_cols") is not None:
                df = df.rename(columns=fichier_origine.get("rename_cols"))
                print("  - rename ", fichier_origine.get("rename_cols"))
            # Pour info et mise au point
            #print(f"{nom_fichier:25s}  : {df.shape[0]:6d} observations {df.shape[1]:2d} variables")
            
            print(f"{nom_fichier:25s}  : {df.shape[0]:6d} x {df.shape[1]:2d}")
            dfl.append(df) # ajout du DataFrame annuel à la liste

     # Concaténation de tous les DataFrames annuels (2019->2022)
    dfrub[key] = pd.concat(dfl)

    # Affichages pour mise au point et contrôle
    print("Nombre de DataFrames       : ", len(dfl))
    print("Nombre d'observations      : ", nb_obs, dfrub[key].shape[0])
    print("Colonnes du DataFrame      : ", dfrub[key].shape[1])
    print ()

    dfrub[key].to_csv(rep_dst + key + ".csv", sep='\t', index=False)

# Ces variables permettent d'alléger l'écriture.
# Nota : L'affectation des DataFrames dans de nouvelles variables
# se fait sans recopie de données et ne nécessite pas
# d'allocation importante de mémoire.
 
dfu = dfrub["usagers"]
dfv = dfrub["vehicules"]
dfl = dfrub["lieux"]
dfc = dfrub["caracteristiques"]

print ("Chargement des données en mémoire réalisé.")

 -  Chargement des fichiers

Rubrique :  caracteristiques
caracteristiques-2019.csv  :  58840 x 15
caracteristiques-2020.csv  :  47744 x 15
carcteristiques-2021.csv   :  56518 x 15
  - rename  {'Accident_Id': 'Num_Acc'}
carcteristiques-2022.csv   :  55302 x 15
Nombre de DataFrames       :  4
Nombre d'observations      :  218404 218404
Colonnes du DataFrame      :  15


Rubrique :  usagers
usagers-2019.csv           : 132977 x 15
usagers-2020.csv           : 105295 x 15
usagers-2021.csv           : 129248 x 16
usagers-2022.csv           : 126662 x 16
Nombre de DataFrames       :  4
Nombre d'observations      :  494182 494182
Colonnes du DataFrame      :  16


Rubrique :  vehicules
vehicules-2019.csv         : 100710 x 11
vehicules-2020.csv         :  81066 x 11
vehicules-2021.csv         :  97315 x 11
vehicules-2022.csv         :  94493 x 11
Nombre de DataFrames       :  4
Nombre d'observations      :  373584 373584
Colonnes du DataFrame      :  11


Rubrique :  lieux
lieux-2019.csv    

In [2]:
phase = "_21" # Ajouté aux noms de fichiers pour distinguer la phase


In [3]:

# Ici nous forçons la conversion du champ Num_Acc en entier
# pour pouvoir faire les jointures.
# TODO : jeter un coup d'œil à num_ACC et contrôler
log_action("Conversion des champs Num_Acc en type entier")
dfc.Num_Acc = dfc.Num_Acc.astype(int)
dfu.Num_Acc = dfu.Num_Acc.astype(int)
dfl.Num_Acc = dfl.Num_Acc.astype(int)
dfv.Num_Acc = dfv.Num_Acc.astype(int)


 -  Conversion des champs Num_Acc en type entier


In [4]:
########################################################################
# Jointures des 4 rubriques 
########################################################################

# Lesjointures sont toutes faites avec le champ Num_Acc 
# La dernière jointure avec les véhicules est faite avec, en plus,
# les champs num_veh et id_vehicule.
# Les jointures sont "à gauche" ("left")
# pour conserver le nombre d'usagers.
# Les nombre d'observations et de variables affichées
# avant et après jointure permettent de les vérifier.
# Il y a 494 182 usagers avant les jointures et le DataFrame résultant
# a ce même nombre d'observations.

print("")
log_action("Jointure (gauche) usagers  <--->  caracteristiques avec NumAcc")
#print("Un pour un")
print ()
print("Taille usagers          : ", dfu.shape)
print("Taille caractéristiques : ", dfc.shape)
df = pd.merge(dfu, dfc, on="Num_Acc", how = "left")
print("Tailles df résultant    : ", df.shape)

print("")
log_action("Jointure gauche (usagers et caracteristiques)  <--->  lieux avec NumAcc")
print()
print("Taille DataFrame        : ", df.shape)
print("Taille lieux            : ", dfl.shape)
df = pd.merge(df, dfl, on="Num_Acc", how = "left")
print("Tailles df résultant    : ", df.shape)

print("")
log_action("Jointure (usagers, caracteristiques et lieux)  <--->  vehicules avec NumAcc, id_vehicule et num_veh")
print()
print("Taille DataFrame        : ", df.shape)
print("Taille vehicules        : ", dfv.shape)
df = pd.merge(df, dfv, on=["Num_Acc", "id_vehicule", "num_veh"], how = "left")
print("Tailles df résultant    : ", df.shape)
print ()
print ("-----------------------------------------------------------------------")
print ()
print ("Colonnes : ", df.columns)




 -  Jointure (gauche) usagers  <--->  caracteristiques avec NumAcc

Taille usagers          :  (494182, 16)
Taille caractéristiques :  (218404, 15)
Tailles df résultant    :  (494182, 30)

 -  Jointure gauche (usagers et caracteristiques)  <--->  lieux avec NumAcc

Taille DataFrame        :  (494182, 30)
Taille lieux            :  (218404, 18)
Tailles df résultant    :  (494182, 47)

 -  Jointure (usagers, caracteristiques et lieux)  <--->  vehicules avec NumAcc, id_vehicule et num_veh

Taille DataFrame        :  (494182, 47)
Taille vehicules        :  (373584, 11)
Tailles df résultant    :  (494182, 55)

-----------------------------------------------------------------------

Colonnes :  Index(['Num_Acc', 'id_vehicule', 'num_veh', 'place', 'catu', 'grav', 'sexe',
       'an_nais', 'trajet', 'secu1', 'secu2', 'secu3', 'locp', 'actp', 'etatp',
       'id_usager', 'jour', 'mois', 'an', 'hrmn', 'lum', 'dep', 'com', 'agg',
       'int', 'atm', 'col', 'adr', 'lat', 'long', 'catr', 'voie', 

In [5]:
col_gardees = { 'place', 'catu', 'grav', 'sexe',
       'an_nais', 'trajet', 'secu1', 'secu2', 'secu3', 'locp', 'actp', 'etatp',
       'jour', 'mois', 'an', 'hrmn', 'lum', 'dep',  'agg',
       'int', 'atm', 'col', 'catr',
       'circ', 'nbv', 'vosp', 'prof',  'plan',
       'surf', 'infra', 'situ', 'vma', 'senc', 'catv', 'obs', 'obsm', 'choc',
       'manv', 'motor'}

col_supprimees = df.columns.difference(col_gardees)
log_action ("Suppression des colonnes :\n" + str(col_supprimees))

df = df.drop (col_supprimees, axis=1)
print ("Il reste les colonnes suivantes :\n", df.columns)
print ("Recherche des valeurs nulles")
print ("Valeurs nulles :\n", df.isnull().sum())


 -  Suppression des colonnes :
Index(['Num_Acc', 'adr', 'com', 'id_usager', 'id_vehicule', 'larrout',
       'lartpc', 'lat', 'long', 'num_veh', 'occutc', 'pr', 'pr1', 'v1', 'v2',
       'voie'],
      dtype='object')
Il reste les colonnes suivantes :
 Index(['place', 'catu', 'grav', 'sexe', 'an_nais', 'trajet', 'secu1', 'secu2',
       'secu3', 'locp', 'actp', 'etatp', 'jour', 'mois', 'an', 'hrmn', 'lum',
       'dep', 'agg', 'int', 'atm', 'col', 'catr', 'circ', 'nbv', 'vosp',
       'prof', 'plan', 'surf', 'infra', 'situ', 'vma', 'senc', 'catv', 'obs',
       'obsm', 'choc', 'manv', 'motor'],
      dtype='object')
Recherche des valeurs nulles
Valeurs nulles :
 place         0
catu          0
grav          0
sexe          0
an_nais    5941
trajet        0
secu1         0
secu2         0
secu3         0
locp          0
actp          0
etatp         0
jour          0
mois          0
an            0
hrmn          0
lum           0
dep           0
agg           0
int           0
atm      

In [6]:
# La seule colonne contenant des nulls est celle de l'année de naissance (an_nais)
# Les valeurs nulles sont remplacées par 0
log_action(f"Remplacement des nulls ({df.an_nais.isna().sum()} valeurs)de l'année de naissance (an_nais) par 0")
df.an_nais = df.an_nais.fillna(0)


# Suppression des minutes dans l'heure
df.hrmn = df.hrmn.str[:2]
df.hrmn = df.hrmn.astype(int)
print ("Répartition des heures")
print (df.hrmn.value_counts())
log_action (f"Suppression des minutes dans la variable hrmn (heure) et conversion en entier ")


# Conversion de chaque colonne en entier.
# Des valeurs non disponibles sont marquées " -1" et forcent
# la conversion en objet.
# Il y a des colonnes avec du texte
col_non_converties = []
for col in df.columns :
    try :
        df[col] = df[col].astype(int)
    except (ValueError, TypeError):
        #print (f"Colonne {col} : conversion non réalisée, mais ce n'est pas grave.")
        col_non_converties.append(col)
        pass

log_action (f"Conversion des colonnes en entiers (int64) sauf {col_non_converties}")


 -  Remplacement des nulls (5941 valeurs)de l'année de naissance (an_nais) par 0
Répartition des heures
hrmn
17    44785
18    42452
16    37304
19    31638
15    31084
14    27929
8     27310
12    26012
13    25039
11    24697
9     23378
10    22305
20    21708
7     20610
21    15698
22    12495
23    10861
6      9904
0      9054
1      7224
5      6614
2      6320
3      4921
4      4840
Name: count, dtype: int64
 -  Suppression des minutes dans la variable hrmn (heure) et conversion en entier 
 -  Conversion des colonnes en entiers (int64) sauf ['actp', 'dep', 'nbv']


In [7]:
nb = df.shape[0]
#df[df.duplicated()].sort_values(by=["an"])
df = df.drop_duplicates()

log_action (f"Suppression de {(nb - df.shape[0])} observation pour éliminer les doublons.")


 -  Suppression de 469 observation pour éliminer les doublons.


In [8]:
# Réalisation d'un rapport automatique avec statistiques exploratoires de base.
# C'est grossier mais permet d'avoir rappidement un bon aperçu des données.
"""
fic_rapport = open(rep_dst+"data_pour_modelisation.txt", 'w')
#print(rep_dst+"desc_vars.json") # Pour mise au point
premier_rapport(df,
                max_list_len=30,
                csv_desc_name=rep_dst+"desc_vars.json",
                outfile=fic_rapport)
fic_rapport.close()
"""
# et affichage des infos pour contrôle
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 493713 entries, 0 to 494181
Data columns (total 39 columns):
 #   Column   Non-Null Count   Dtype 
---  ------   --------------   ----- 
 0   place    493713 non-null  int64 
 1   catu     493713 non-null  int64 
 2   grav     493713 non-null  int64 
 3   sexe     493713 non-null  int64 
 4   an_nais  493713 non-null  int64 
 5   trajet   493713 non-null  int64 
 6   secu1    493713 non-null  int64 
 7   secu2    493713 non-null  int64 
 8   secu3    493713 non-null  int64 
 9   locp     493713 non-null  int64 
 10  actp     493713 non-null  object
 11  etatp    493713 non-null  int64 
 12  jour     493713 non-null  int64 
 13  mois     493713 non-null  int64 
 14  an       493713 non-null  int64 
 15  hrmn     493713 non-null  int64 
 16  lum      493713 non-null  int64 
 17  dep      493713 non-null  object
 18  agg      493713 non-null  int64 
 19  int      493713 non-null  int64 
 20  atm      493713 non-null  int64 
 21  col      493713

In [9]:

phase = "_11" # Ajouté aux noms de fichiers pour distinguer la phase
df.to_csv(rep_dst+"data"+phase+".csv", sep='\t', index=False)

df_bak = df # pour récupération pendant codage 


# Réduction de données
Nous Réalisons une réduction du nombre de 
La variable cible est la gravité pour l'usager (grav).
Elle a quatre modalités inégalement réparties et des valeurs manquantes.
Nous supprimons les observations pour lesquelles cette gravité est inconnue
et nous sous-échantillonons les modalités.
Nous limitons à 1500 le nombre d'échantillons.



In [10]:
df1 = df

print ("Répartition des données selon la gravité : ")

nb_moins_1 = df1[df1.grav == -1].shape[0]

log_action (f"Suppression de {df1[df1.grav == -1].shape[0]} observation dont la gravité est inconnue (valeur -1)")
df1 = df1[df.grav != -1]
print (df1.grav.value_counts())
min_count  = df1.grav.value_counts().min()

log_action (f"Sous-échantillonnage à {min_count} observations de chacune des 4 modalité de gravité")
df_g1 = df1[df1.grav == 1].sample(min_count)
df_g2 = df1[df1.grav == 2].sample(min_count)
df_g3 = df1[df1.grav == 3].sample(min_count)
df_g4 = df1[df1.grav == 4].sample(min_count)

df1 = pd.concat([df_g1, df_g2, df_g3, df_g4], ignore_index = True)

print ("Répartition des modalités de la gravité après sous-échantillonage")
print (df1.grav.value_counts())
df = df1


Répartition des données selon la gravité : 
 -  Suppression de 298 observation dont la gravité est inconnue (valeur -1)
grav
1    207111
4    197302
3     75957
2     13045
Name: count, dtype: int64
 -  Sous-échantillonnage à 13045 observations de chacune des 4 modalité de gravité
Répartition des modalités de la gravité après sous-échantillonage
grav
1    13045
2    13045
3    13045
4    13045
Name: count, dtype: int64


In [11]:
# Enregistrement du DataFrameRésultant

log_action("Enregistrement du jeu de données")
df.to_csv(rep_dst+"data_modelisation.csv", sep = '\t')
          


 -  Enregistrement du jeu de données


In [12]:
print ("Récapitulatif des actions réalisées.")
for a in actions:
    print (a)

Récapitulatif des actions réalisées.
Chargement des fichiers
Conversion des champs Num_Acc en type entier
Jointure (gauche) usagers  <--->  caracteristiques avec NumAcc
Jointure gauche (usagers et caracteristiques)  <--->  lieux avec NumAcc
Jointure (usagers, caracteristiques et lieux)  <--->  vehicules avec NumAcc, id_vehicule et num_veh
Suppression des colonnes :
Index(['Num_Acc', 'adr', 'com', 'id_usager', 'id_vehicule', 'larrout',
       'lartpc', 'lat', 'long', 'num_veh', 'occutc', 'pr', 'pr1', 'v1', 'v2',
       'voie'],
      dtype='object')
Remplacement des nulls (5941 valeurs)de l'année de naissance (an_nais) par 0
Suppression des minutes dans la variable hrmn (heure) et conversion en entier 
Conversion des colonnes en entiers (int64) sauf ['actp', 'dep', 'nbv']
Suppression de 469 observation pour éliminer les doublons.
Suppression de 298 observation dont la gravité est inconnue (valeur -1)
Sous-échantillonnage à 13045 observations de chacune des 4 modalité de gravité
Enregist